# 05 · Quantum machine learning with PyTorch

Combine an ordinary PyTorch optimizer with `fq.Module`. The same module can later select another runtime through `fq.RuntimePolicy`.

In [ ]:
import torch
import flagquantum as fq

def classifier_circuit(parameters, inputs):
    circuit = fq.Circuit(n_qubits=1, bsz=inputs.shape[0])
    circuit.ry(qubit=0, theta=inputs + parameters['bias'])
    return circuit

model = fq.Module(
    classifier_circuit,
    parameters={'bias': ()},
    init={'bias': 0.5},
    policy=fq.RuntimePolicy(observable='z', observable_wires=(0,)),
)
features = torch.tensor([-1.0, -0.5, 0.5, 1.0])
targets = torch.sign(features)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)

In [ ]:
for _ in range(20):
    optimizer.zero_grad()
    prediction = model(features)
    loss = torch.nn.functional.mse_loss(prediction, targets)
    loss.backward()
    optimizer.step()

print({'loss': float(loss.detach()), 'runtime': model.execute(features).runtime})